In [13]:
!pip install onnxruntime onnxruntime-gpu keras-image-helper

In [9]:
!wget https://github.com/DataTalksClub/machine-learning-zoomcamp/releases/download/dl-models/clothing-model-new.onnx -O clothing-model-new.onnx

--2026-02-19 13:52:26--  https://github.com/DataTalksClub/machine-learning-zoomcamp/releases/download/dl-models/clothing-model-new.onnx
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/256401220/62905e85-e051-4260-8cba-c18dd0b621ea?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-19T14%3A38%3A36Z&rscd=attachment%3B+filename%3Dclothing-model-new.onnx&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-19T13%3A38%3A21Z&ske=2026-02-19T14%3A38%3A36Z&sks=b&skv=2018-11-09&sig=B46OupQ9I4SX3t%2BHLB4oK7ZN7cCWxtidH3mkapvV0zE%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MTUxMDg0NSwibmJmIjoxNzcxNTA5MDQ1LCJwY

In [10]:
import onnxruntime as ort


onnx_model_path = 'clothing-model-new.onnx'
session = ort.InferenceSession(onnx_model_path, providers=['CUDAExecutionProvider'])

inputs = session.get_inputs()
outputs = session.get_outputs()

input_name = inputs[0].name
output_name = outputs[0].name

In [12]:
input_name, output_name

('input_layer_6', 'output_0')

In [14]:
test_url = 'http://bit.ly/mlbookcamp-pants'

In [16]:
# We use keras-image-helper to download and preprocess the image
# This is useful to avoid a dependency on the whole TensorFlow which is huge

from keras_image_helper import create_preprocessor

preprocessor = create_preprocessor('xception', target_size=(299, 299))
                                   
X = preprocessor.from_url(test_url)

X

array([[[[-0.11372548, -0.15294117, -0.19999999],
         [-0.11372548, -0.15294117, -0.19999999],
         [-0.10588235, -0.14509803, -0.19215685],
         ...,
         [-0.01960784, -0.01960784, -0.08235294],
         [-0.04313725, -0.04313725, -0.10588235],
         [-0.11372548, -0.11372548, -0.17647058]],

        [[-0.09019607, -0.12941176, -0.17647058],
         [-0.09019607, -0.12941176, -0.17647058],
         [-0.08235294, -0.12156862, -0.16862744],
         ...,
         [-0.01960784, -0.01960784, -0.08235294],
         [-0.04313725, -0.04313725, -0.10588235],
         [-0.10588235, -0.10588235, -0.16862744]],

        [[-0.09803921, -0.1372549 , -0.18431371],
         [-0.09803921, -0.1372549 , -0.18431371],
         [-0.09019607, -0.12941176, -0.17647058],
         ...,
         [-0.01960784, -0.01960784, -0.08235294],
         [-0.03529412, -0.03529412, -0.09803921],
         [-0.09019607, -0.09019607, -0.15294117]],

        ...,

        [[-0.67058825, -0.7019608 , -0

In [23]:
preds = session.run([output_name], {input_name: X})

float_predictions = preds[0][0].tolist()

float_predictions

[-2.3684329986572266,
 -4.414735794067383,
 -2.309055805206299,
 -1.739748239517212,
 8.660110473632812,
 -3.194986581802368,
 -5.595435619354248,
 2.7487964630126953,
 -2.995649814605713,
 -4.363430500030518]

In [25]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

dict(zip(classes, float_predictions))

{'dress': -2.3684329986572266,
 'hat': -4.414735794067383,
 'longsleeve': -2.309055805206299,
 'outwear': -1.739748239517212,
 'pants': 8.660110473632812,
 'shirt': -3.194986581802368,
 'shoes': -5.595435619354248,
 'shorts': 2.7487964630126953,
 'skirt': -2.995649814605713,
 't-shirt': -4.363430500030518}